####  학습목표
- 새로운 내용보다는 기존 내용을 다시 학습하면서 LLM, RAG, Langchain 기초 다지기


In [1]:
'''
LLM : 문장을 이해하고 답변할 수 있는 인공지능모델(알고리즘)  
RAG(Retrieval Augmented Genration) : LLM이 대답하기 전에 관련 문서를 찾아서 참고할 수 있도록 도와주는 비
LangChain : LLM + RAG - AI 연결 조립 키트(LLM, 검색, DB, 체인을 연결해서 자동화 프로그램을 만들 수 있다.) 
'''

'\nLLM : 문장을 이해하고 답변할 수 있는 인공지능모델(알고리즘)  \nRAG(Retrieval Augmented Genration) : LLM이 대답하기 전에 관련 문서를 찾아서 참고할 수 있도록 도와주는 비\nLangChain : LLM + RAG - AI 연결 조립 키트(LLM, 검색, DB, 체인을 연결해서 자동화 프로그램을 만들 수 있다.) \n'

In [2]:
import os
import openai
##############################
from   openai import OpenAI
from   dotenv import load_dotenv

In [9]:
# .env api key load (보안상 안전을 위해서 마스킹) 
load_dotenv()
api_key = os.getenv('OPENAI_API_KEY') 

def masking(key) :
    # 앞 4자리, 뒤 4자리만 남기고 * 처리 
    if len(key) <= 8 :
        return '*' * len(key)
    return  key[ : 4 ] + "*" * (len(key) - 8 ) + key[ -4 : ] 
    
masked_api_key = masking(api_key)
print('masked api key : ' , masked_api_key) 

masked api key :  sk-p************************************************************************************************************************************************************dv0A


In [19]:
print('LLM - ')
client = OpenAI(api_key=api_key)

prompt = input('검색하고자하는 내용을 입력하세요 : ')
print('prompt - ' , prompt )

# model : gpt-3.5-turbo - 채팅용 최적화된 가성비 모델
# model : gpt-4o - 텍스트와 이미지/비전 지원을 도와주는 멀티 모델
# model : gpt-4o-mini - 속도/비용 측면에서 유리 

system_content='''
당신은 친절한 파이썬 보안 도우미입니다. 
사용자의 요청에 대해 항상 보안 모범 사례를 우선으로 설명하고, 
민감 정보 노출을 방지하는 방법, 최소 권한 원칙, 패키지/채널 검증, 파일 권한 설정, 
취약점 완화 방법을 구체적 명령어와 체크리스트 형태로 제공하십시오. 
응답에 실제 비밀번호나 실사용 API 키를 절대 포함하지 마십시오. 
'''
user_content=f'''
1) 패키지 설치시 보안 지침
2) 모니터링 권장 설정 방법
3) 민감정보 관리 방법과 예시
4) 가상환경 구축 권장방법 {prompt}'''

response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        # role : system, user, assistance
        # content : content
        {'role' : 'system', 'content' : system_content},
        {'role' : 'user'  , 'content' : user_content},
    ],
    # 응답 문장의 길이제한
    max_tokens=512,
    # 출력 다양성(무작위성) : 보수적, 창의적 : 0 ~ 1 낮을수록 보수적
    temperature=0.8
)
print('response - ')
print(response)
print()
print('content - ')
print(response.choices[0].message.content)

LLM - 


검색하고자하는 내용을 입력하세요 :  코드보안


prompt -  코드보안
response - 
ChatCompletion(id='chatcmpl-CaXqMX8myXQxM4lzdBcPtam67oKTO', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='안녕하세요! 보안 모범 사례를 준수하며 요청하신 내용을 자세히 설명드리겠습니다.\n\n### 1) 패키지 설치시 보안 지침\n\n- **패키지 검증**: 공식 패키지 저장소(Pypi 등)에서 패키지를 설치합니다.\n- **버전 고정**: 패키지의 버전을 명시하여 예기치 않은 업데이트로 인한 취약점을 방지합니다.\n  ```bash\n  pip install package_name==1.2.3\n  ```\n- **서명 검증**: 패키지가 서명된 경우, 서명을 확인하여 무결성을 유지합니다.\n- **보안 스캐너 사용**: `safety`와 같은 도구를 사용하여 패키지의 취약점을 스캔합니다.\n  ```bash\n  pip install safety\n  safety check\n  ```\n\n### 2) 모니터링 권장 설정 방법\n\n- **로그 기록**: 모든 활동과 오류에 대해 로깅합니다. 다음은 기본적인 로깅 설정 예시입니다.\n  ```python\n  import logging\n\n  logging.basicConfig(level=logging.INFO, \n                      filename=\'app.log\', \n                      format=\'%(asctime)s - %(levelname)s - %(message)s\')\n  ```\n\n- **모니터링 도구 설정**: `Prometheus`, `Grafana`, `ELK Stack`과 같은 도구를 설정하여 시스템 상태를 실시간으로 감시합니다.\n\n- **알림 설정**: 이상 감지 시 즉시 알림을 받을 수 있도록

In [53]:
'''
도서관 사서 챗봇 시나리오  
- 사용자 묻고
- 챗봇(인공지능모델)은 먼저 DB(RAG) 관련 내용을 찾아본 후
- 그 정보를 참고해서 똑똑해진 후 
- 사용자에게 응답 
'''

from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores      import FAISS
from langchain.text_splitter     import CharacterTextSplitter
from langchain.chat_models       import ChatOpenAI 
from langchain.chains            import RetrievalQA

In [22]:
# 외부 DB 개념으로 
doc = [
    '리스트는 파이썬에서 변경 가능한(mutable) 자료형으로, 요소를 추가하거나 삭제할 수 있습니다.',
    '튜플은 변경 불가능한(immutable) 자료형으로, 한 번 생성하면 수정할 수 없습니다.',
    '딕셔너리는 키(key)와 값(value)의 쌍으로 데이터를 저장합니다.'
]

In [41]:
# RAG 숫자배열로 변환
# chunk_size 는 chunk_overlap(겹치는 글자 수) 보다 커야함 

text_splitter = CharacterTextSplitter(chunk_size=250)
texts = text_splitter.create_documents(doc)
print(texts)

[Document(page_content='리스트는 파이썬에서 변경 가능한(mutable) 자료형으로, 요소를 추가하거나 삭제할 수 있습니다.'), Document(page_content='튜플은 변경 불가능한(immutable) 자료형으로, 한 번 생성하면 수정할 수 없습니다.'), Document(page_content='딕셔너리는 키(key)와 값(value)의 쌍으로 데이터를 저장합니다.')]


In [43]:
# embedding & RAG
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
db = FAISS.from_documents( texts , embedding = embeddings)
print(db)

In [44]:
# 검색
# as_retriever() : 검색 인터페이스를 이용해서 LLM 연결하는 것 
# Retriever 설정
retriever = db.as_retriever(search_kwargs={'k' : 1}) # 반환 문서 수 : 1
print(retriever)


tags=['FAISS', 'OpenAIEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000027F2DA29750> search_kwargs={'k': 1}


In [45]:
# chain
# SystemMessage, HumanMessage

qa = RetrievalQA.from_chain_type(
    llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.9),
    # stuff, map_reduce, refine etc....
    chain_type='stuff',
    retriever = retriever
)
print(qa)

combine_documents_chain=StuffDocumentsChain(llm_chain=LLMChain(prompt=ChatPromptTemplate(input_variables=['context', 'question'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], template="Use the following pieces of context to answer the user's question. \nIf you don't know the answer, just say that you don't know, don't try to make up an answer.\n----------------\n{context}")), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], template='{question}'))]), llm=ChatOpenAI(client=<openai.resources.chat.completions.Completions object at 0x0000027F2DA2A530>, async_client=<openai.resources.chat.completions.AsyncCompletions object at 0x0000027F2DA40E50>, model_name='gpt-4o-mini', temperature=0.9, openai_api_key='YOUR_API_KEY', openai_proxy='')), document_variable_name='context') retriever=VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000027F2DA

In [51]:
# 질의 : 문서기반질의(RAG가 강한 부분)
'''
간단한 사실 확인
예제코드 요청
비교-선택 도움

문서기반질의(RAG가 강한 부분)

'''
query  = '파이썬 리스트와 튜플의 차이점을 설명해줘'
answer = qa.run(query)

print('Q - ' , query) 
print()
print('사서가 참고한 내용 - ' , retriever.get_relevant_documents(query)[0].page_content )
print()
print('A - ' , answer)

Q -  파이썬 리스트와 튜플의 차이점을 설명해줘

사서가 참고한 내용 -  리스트는 파이썬에서 변경 가능한(mutable) 자료형으로, 요소를 추가하거나 삭제할 수 있습니다.

A -  파이썬 리스트와 튜플의 주요 차이점은 다음과 같습니다:

1. **변경 가능성**: 리스트는 변경 가능한(mutable) 자료형으로, 요소를 추가하거나 삭제할 수 있습니다. 반면에 튜플은 변경 불가능한(immutable) 자료형으로, 한 번 생성하면 그 내용을 변경할 수 없습니다.

2. **구성**: 리스트는 대괄호([])로 정의하고, 튜플은 소괄호(())로 정의합니다.

3. **성능**: 튜플은 리스트보다 일반적으로 메모리 사용량이 적고, 성능이 좋습니다. 이는 튜플이 불변성을 가지기 때문에 더 효율적으로 저장될 수 있습니다.

4. **용도**: 리스트는 데이터에 대한 변경이 필요할 때 주로 사용되며, 튜플은 데이터가 고정되어 있어야 할 때, 즉 변경할 필요가 없거나 변경해서는 안 되는 경우에 사용됩니다. 

이러한 차이로 인해, 리스트와 튜플은 서로 다른 상황에서 사용됩니다.
